In [1]:
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import models

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report, roc_curve, auc, precision_recall_curve
)
from sklearn.feature_extraction.text import TfidfVectorizer


In [2]:
#Makes it produce the same results after every training
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)


In [4]:
#Get data
CSV_PATH = "Phishing_Email.csv"
LABEL_COL = "Email Type"        # target column
TEXT_COL = "Email Text"         # email content


In [5]:
#pandas datafram
df = pd.read_csv(CSV_PATH)

In [6]:
# Clean column names
df.columns = df.columns.str.strip()

# Drop unnecessary index column if exists
if "Unnamed: 0" in df.columns:
    df = df.drop(columns=["Unnamed: 0"])

# Drop missing labels
df = df.dropna(subset=[LABEL_COL])

In [ ]:
# Encode labels
le = LabelEncoder()
y = le.fit_transform(df[LABEL_COL])  # Safe Email -> 0, Phishing Email -> 1

In [8]:
#convert text to idf tf
#(Term Frequency–Inverse Document Frequency)
#1000 max important words
vectorizer = TfidfVectorizer(max_features=1000)
X_text = vectorizer.fit_transform(df[TEXT_COL].fillna('')).toarray()


In [9]:

X_final = X_text

In [10]:
#scale
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_final)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=SEED, stratify=y
)
'''
print("Label distribution (train):")
print(pd.Series(y_train).value_counts())
print("Label distribution (test):")
print(pd.Series(y_test).value_counts())
'''

Label distribution (train):
1    9058
0    5862
Name: count, dtype: int64
Label distribution (test):
1    2264
0    1466
Name: count, dtype: int64


In [ ]:
#make class weigths since slight data imbalance
unique_classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=unique_classes, y=y_train)
class_weights = {cls: w for cls, w in zip(unique_classes, weights)}
print("Class Weights:", class_weights)

Class Weights: {np.int64(0): np.float64(1.272603207096554), np.int64(1): np.float64(0.8235813645396335)}


In [13]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

c:\Users\ajp12\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [14]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)
#model.summary()

In [16]:
#stop early if val loss doesnt descrease after 3 epochs
es = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = model.fit(
    X_train, y_train.astype(np.float32),
    validation_split=0.2,
    epochs=5,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[es],
    verbose=1
)

Epoch 1/5
373/373 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9597 - auc: 0.9923 - loss: 0.1034 - val_accuracy: 0.9460 - val_auc: 0.9886 - val_loss: 0.1293
Epoch 2/5
373/373 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9700 - auc: 0.9962 - loss: 0.0705 - val_accuracy: 0.9491 - val_auc: 0.9873 - val_loss: 0.1460
Epoch 3/5
373/373 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9782 - auc: 0.9981 - loss: 0.0479 - val_accuracy: 0.9481 - val_auc: 0.9836 - val_loss: 0.1738
Epoch 4/5
373/373 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - accuracy: 0.9810 - auc: 0.9984 - loss: 0.0431 - val_accuracy: 0.9457 - val_auc: 0.9806 - val_loss: 0.2114


In [ ]:
#only run this when you want to save a model
'''
# Use raw string (r"...") or double backslashes to avoid escape issues
import os
#Use raw string r"" to avoid escape issues
#model.save(r"YOURPATH\phishing_model.keras")


save_path = r"YOUTPATH\phishing_model.keras"
# Make sure folder exists
os.makedirs(os.path.dirname(save_path), exist_ok=True)

model.save(save_path)
print("Model ")
'''

In [17]:
#evaluate the model accuracy
eval_res = model.evaluate(X_test, y_test.astype(np.float32), verbose=0)
print("\nTest results [loss, accuracy, auc]:", eval_res)


Test results [loss, accuracy, auc]: [0.11228231340646744, 0.9536193013191223, 0.9916012287139893]


In [18]:
#predict probabilities
y_probs = model.predict(X_test).ravel()
threshold = 0.5
y_pred = (y_probs > threshold).astype(int)

117/117 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
